In [2]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import json

print("Imports OK")

Imports OK


In [3]:
with open("ensa_kb.json", "r", encoding="utf-8") as f:
    ensa_data = json.load(f)

# Extraire les textes à indexer (question + réponse combinées)
passages = []
for item in ensa_data:
    texte = item["question"] + " " + item["answer"]
    passages.append(texte)

print(f"{len(passages)} passages chargés")
print("Exemple :", passages[0][:150])

320 passages chargés
Exemple : How can I get admitted to ENSA Fès? Admission to ENSA Fès is done through the National Common Entrance Exam (CNC) for scientific baccalaureate holders


In [5]:
print("Chargement du modèle... (1-2 minutes la première fois)")
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Modèle chargé !")

Chargement du modèle... (1-2 minutes la première fois)


C:\Users\BrandlyFES\anaconda3\envs\DLxNLP\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Modèle chargé !


In [6]:
print("Encodage des passages...")
embeddings = model.encode(passages, show_progress_bar=True)

print("Shape des embeddings :", embeddings.shape)
# Tu dois voir : (nombre_entrées, 384)

Encodage des passages...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Shape des embeddings : (320, 384)


In [7]:
dimension = embeddings.shape[1]  # 384
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype(np.float32))

print(f"Index FAISS construit : {index.ntotal} vecteurs indexés")

Index FAISS construit : 320 vecteurs indexés


In [8]:
def retrieve(query, top_k=3):
    # Encoder la question
    query_embedding = model.encode([query]).astype(np.float32)
    
    # Chercher les passages les plus proches
    distances, indices = index.search(query_embedding, top_k)
    
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "rank": i + 1,
            "score": float(distances[0][i]),
            "question": ensa_data[idx]["question"],
            "answer": ensa_data[idx]["answer"],
            "category": ensa_data[idx]["category"]
        })
    return results

In [9]:
# Test 1
print("=== TEST 1 ===")
resultats = retrieve("Comment je peux entrer à l'ENSA ?")
for r in resultats:
    print(f"[Rank {r['rank']}] Score: {r['score']:.2f} | {r['question']}")
    print(f"  Réponse: {r['answer'][:100]}...")
    print()

# Test 2
print("=== TEST 2 ===")
resultats2 = retrieve("quels clubs existe à l'école ?")
for r in resultats2:
    print(f"[Rank {r['rank']}] Score: {r['score']:.2f} | {r['question']}")
    print()

# Test 3
print("=== TEST 3 ===")
resultats3 = retrieve("c'est quoi les débouchés après l'ENSA ?")
for r in resultats3:
    print(f"[Rank {r['rank']}] Score: {r['score']:.2f} | {r['question']}")
    print()

=== TEST 1 ===
[Rank 1] Score: 1.34 | What is the language of instruction at ENSA Fès?
  Réponse: The primary language of instruction at ENSA Fès is French, which is the standard for technical and s...

[Rank 2] Score: 1.37 | What language is the ENSA Fès PFE report written in?
  Réponse: PFE reports at ENSA Fès are primarily written in French, with an abstract (résumé) required in both ...

[Rank 3] Score: 1.41 | What is the stage ouvrier internship report format at ENSA Fès?
  Réponse: The worker internship (stage ouvrier) report is a short document (typically 15–30 pages) describing ...

=== TEST 2 ===
[Rank 1] Score: 1.05 | How can I join a club at ENSA Fès?

[Rank 2] Score: 1.09 | What student clubs are available at ENSA Fès?

[Rank 3] Score: 1.10 | What is the Clubs Day at ENSA Fès?

=== TEST 3 ===
[Rank 1] Score: 1.42 | What is USMBA and what is its relation to ENSA Fès?

[Rank 2] Score: 1.42 | Who are the faculty members of the SDSC department at ENSA Fès?

[Rank 3] Score: 1.43

In [10]:
faiss.write_index(index, "ensa_faiss.index")

# Sauvegarder aussi les passages pour les retrouver par indice
with open("passages.json", "w", encoding="utf-8") as f:
    json.dump(passages, f, ensure_ascii=False, indent=2)

print("Index FAISS sauvegardé : ensa_faiss.index")
print("Passages sauvegardés : passages.json")

Index FAISS sauvegardé : ensa_faiss.index
Passages sauvegardés : passages.json
